In [1]:
from pathlib import Path
import pandas as pd

# ==========================================================
# Load and Prepare Data
# ==========================================================
# Load cleaned datasets from processed folder
processed_dir = Path("data/processed")
sales_clean = pd.read_csv(processed_dir / "sales_clean.csv")
future_clean = pd.read_csv(processed_dir / "future_clean.csv")

# Convert date columns to datetime
sales_clean["date"] = pd.to_datetime(sales_clean["date"])
future_clean["date"] = pd.to_datetime(future_clean["date"])



C:\Users\setue\AppData\Local\Temp\ipykernel_15540\3603867740.py:9: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  sales_clean = pd.read_csv(processed_dir / "sales_clean.csv")


In [2]:
# ==========================================================
# Feature Engineering
# ==========================================================
# Machine learning models cannot directly understand dates.
# We transform the date into calendar numerical features
# to help the model learn weekly and seasonal sales patterns.
# ==========================================================

# Copy datasets to avoid modifying the cleaned data
sales_fe = sales_clean.copy()
future_fe = future_clean.copy()

In [3]:
# ==========================================================
# Clean state_holiday categories
# ==========================================================

sales_fe["state_holiday"] = (
    sales_fe["state_holiday"]
    .astype(str)
    .replace({
        "0.0": "0",
        "0.1": "0"
    })
)

future_fe["state_holiday"] = (
    future_fe["state_holiday"]
    .astype(str)
    .replace({
        "0.0": "0",
        "0.1": "0"
    })
)

In [4]:
# ==========================================================
# Calendar Features
# ==========================================================
# Machine learning models cannot directly use datetime
# variables. Therefore, we extract calendar-based features
# to capture weekly, monthly and yearly seasonal patterns.
#
# Features:
# year         : Calendar year
# month        : Month of the year (1-12)
# quarter      : Quarter of the year (1-4)
# week_of_year : ISO week number (1-52)
# day_of_week  : Day of the week (Monday=0, Sunday=6)
# day_of_month : Day of the month (1-31)
# is_weekend   : Weekend indicator (0=Weekday, 1=Weekend)
# ==========================================================

calendar_features = [
    "year",
    "month",
    "quarter",
    "week_of_year",
    "day_of_week",
    "day_of_month",
    "is_weekend",
    "is_month_start",
    "is_month_end",
    "is_quarter_end"
]

# ==========================================================
# Generate Calendar Features
# ==========================================================

sales_fe["year"] = sales_fe["date"].dt.year

sales_fe["month"] = sales_fe["date"].dt.month

sales_fe["quarter"] = sales_fe["date"].dt.quarter

sales_fe["week_of_year"] = (
    sales_fe["date"]
    .dt.isocalendar()
    .week
    .astype(int)
)

sales_fe["day_of_week"] = sales_fe["date"].dt.dayofweek

sales_fe["day_of_month"] = sales_fe["date"].dt.day

sales_fe["is_weekend"] = (
        sales_fe["day_of_week"] >= 5
).astype(int)

sales_fe["is_month_start"] = (
    sales_fe["date"]
    .dt.is_month_start
    .astype(int)
)

future_fe["is_month_start"] = (
    future_fe["date"]
    .dt.is_month_start
    .astype(int)
)

sales_fe["is_month_end"] = (
    sales_fe["date"]
    .dt.is_month_end
    .astype(int)
)

future_fe["is_month_end"] = (
    future_fe["date"]
    .dt.is_month_end
    .astype(int)
)

sales_fe["is_quarter_end"] = (
    sales_fe["date"]
    .dt.is_quarter_end
    .astype(int)
)

future_fe["is_quarter_end"] = (
    future_fe["date"]
    .dt.is_quarter_end
    .astype(int)
)


future_fe["year"] = future_fe["date"].dt.year

future_fe["month"] = future_fe["date"].dt.month

future_fe["quarter"] = future_fe["date"].dt.quarter

future_fe["week_of_year"] = (
    future_fe["date"]
    .dt.isocalendar()
    .week
    .astype(int)
)

future_fe["day_of_week"] = future_fe["date"].dt.dayofweek

future_fe["day_of_month"] = future_fe["date"].dt.day

future_fe["is_weekend"] = (
        future_fe["day_of_week"] >= 5
).astype(int)

# ==========================================================
# Validate Calendar Features
# ==========================================================

print("=" * 60)
print("CALENDAR FEATURES")
print("=" * 60)

print("\nTraining Data")
display(
    sales_fe[
        [
            "date",
            *calendar_features
        ]
    ].head()
)

print("\nFuture Data")
display(
    future_fe[
        [
            "date",
            *calendar_features
        ]
    ].head()
)

# ==========================================================
# Check Missing Values
# ==========================================================

print("=" * 60)
print("MISSING VALUES")
print("=" * 60)

print("\nTraining")
print(
    sales_fe[calendar_features]
    .isna()
    .sum()
)

print("\nFuture")
print(
    future_fe[calendar_features]
    .isna()
    .sum()
)

CALENDAR FEATURES

Training Data


,date,year,month,quarter,week_of_year,day_of_week,day_of_month,is_weekend,is_month_start,is_month_end,is_quarter_end
0,2015-07-19,2015,7,3,29,6,19,1,0,0,0
1,2015-07-19,2015,7,3,29,6,19,1,0,0,0
2,2015-07-19,2015,7,3,29,6,19,1,0,0,0
3,2015-07-19,2015,7,3,29,6,19,1,0,0,0
4,2015-07-19,2015,7,3,29,6,19,1,0,0,0



Future Data


,date,year,month,quarter,week_of_year,day_of_week,day_of_month,is_weekend,is_month_start,is_month_end,is_quarter_end
0,2015-09-17,2015,9,3,38,3,17,0,0,0,0
1,2015-09-17,2015,9,3,38,3,17,0,0,0,0
2,2015-09-17,2015,9,3,38,3,17,0,0,0,0
3,2015-09-17,2015,9,3,38,3,17,0,0,0,0
4,2015-09-17,2015,9,3,38,3,17,0,0,0,0


MISSING VALUES

Training
year              0
month             0
quarter           0
week_of_year      0
day_of_week       0
day_of_month      0
is_weekend        0
is_month_start    0
is_month_end      0
is_quarter_end    0
dtype: int64

Future
year              0
month             0
quarter           0
week_of_year      0
day_of_week       0
day_of_month      0
is_weekend        0
is_month_start    0
is_month_end      0
is_quarter_end    0
dtype: int64


In [5]:
# ==========================================================
# Customer Lag Features
# ==========================================================
# Lag features provide historical customer information so
# the model can learn temporal dependencies.
#
# Features:
# customer_lag_1  : Customers from previous day
# customer_lag_7  : Customers from same weekday last week
# customer_lag_14 : Customers from two weeks ago
# customer_lag_21 : Customers from three weeks ago
# customer_lag_28 : Customers from four weeks ago
#
# Customer lag features are among the most important
# predictors for customer forecasting models.
# ==========================================================

# ==========================================================
# Generate Customer Lag Features
# ==========================================================

# Important: sort by store and date first so each store's
# lag is computed from its own historical observations,
# rather than the previous row in the raw dataset.

sales_fe = (
    sales_fe
    .sort_values(["store_id", "date"])
    .reset_index(drop=True)
)

lag_features = [1, 7, 14, 21, 28]

for lag in lag_features:
    sales_fe[f"customer_lag_{lag}"] = (
        sales_fe
        .groupby("store_id", sort=False)["customers"]
        .shift(lag)
    )

# ==========================================================
# Promotion Lag Features
# ==========================================================
# Promotion history may influence future customer demand.

promo_lags = [1, 7]

for lag in promo_lags:
    sales_fe[f"promo_lag_{lag}"] = (
        sales_fe
        .groupby("store_id")["promo"]
        .shift(lag)
    )

# ==========================================================
# Validate Customer Lag Features
# ==========================================================

lag_columns = [
    f"customer_lag_{lag}"
    for lag in lag_features
]

display(
    sales_fe[
        [
            "store_id",
            "date",
            "customers",
            *lag_columns
        ]
    ].head(35)
)

# ==========================================================
# Check Missing Values
# ==========================================================

print(
    sales_fe[lag_columns]
    .isna()
    .sum()
)

print(
    "\nNon-null customer_lag_1 values:",
    sales_fe["customer_lag_1"].notna().sum()
)

,store_id,date,customers,customer_lag_1,customer_lag_7,customer_lag_14,customer_lag_21,customer_lag_28
0,store_1,2013-01-07,785,NaN,NaN,NaN,NaN,NaN
1,store_1,2013-01-08,654,785.0,NaN,NaN,NaN,NaN
2,store_1,2013-01-09,626,654.0,NaN,NaN,NaN,NaN
3,store_1,2013-01-10,615,626.0,NaN,NaN,NaN,NaN
4,store_1,2013-01-11,592,615.0,NaN,NaN,NaN,NaN
5,store_1,2013-01-12,646,592.0,NaN,NaN,NaN,NaN
6,store_1,2013-01-13,0,646.0,NaN,NaN,NaN,NaN
7,store_1,2013-01-14,616,0.0,785.0,NaN,NaN,NaN
8,store_1,2013-01-15,512,616.0,654.0,NaN,NaN,NaN
9,store_1,2013-01-16,530,512.0,626.0,NaN,NaN,NaN


customer_lag_1       676
customer_lag_7      4732
customer_lag_14     9464
customer_lag_21    14196
customer_lag_28    18928
dtype: int64

Non-null customer_lag_1 values: 623948


In [6]:
# ==========================================================
# Rolling Features: Customer Mean and Std
# ==========================================================
# Lag features capture individual historical observations,
# while rolling features summarize recent customer behaviour.
#
# Features:
# customer_rolling_mean_7  : Average customers over previous 7 days
# customer_rolling_mean_14 : Average customers over previous 14 days
# customer_rolling_mean_21 : Average customers over previous 21 days
# customer_rolling_mean_28 : Average customers over previous 28 days
#
# Rolling statistics help machine learning models capture
# short-term customer trends and smooth daily fluctuations.
# ==========================================================

rolling_windows = [7, 14, 21, 28]

for window in rolling_windows:
    sales_fe[f"customer_rolling_mean_{window}"] = (
        sales_fe
        .groupby("store_id")["customers"]
        .transform(
            lambda x: x.shift(1).rolling(window).mean()
        )
    )

    sales_fe[f"customer_rolling_std_{window}"] = (
        sales_fe
        .groupby("store_id")["customers"]
        .transform(
            lambda x: x.shift(1).rolling(window).std()
        )
    )

# Shift by one day to avoid data leakage.
# Only historical customer information is available
# when predicting the current day.

# ==========================================================
# Promotion Rolling Features
# ==========================================================
# Promotion history may influence future customer traffic.
# We calculate the average promotion frequency over the
# previous 7 and 14 days.

for window in [7, 14]:
    sales_fe[f"promo_mean_{window}"] = (
        sales_fe
        .groupby("store_id")["promo"]
        .transform(
            lambda x:
            x.shift(1)
            .rolling(window)
            .mean()
        )
    )

# ==========================================================
# Validate Rolling Features
# ==========================================================

rolling_columns = []

for window in rolling_windows:
    rolling_columns.extend([
        f"customer_rolling_mean_{window}",
        f"customer_rolling_std_{window}"
    ])

store = "store_1"

display(
    sales_fe.loc[
        sales_fe["store_id"] == store,
        [
            "date",
            "customers",
            *rolling_columns
        ]
    ].head(35)
)

# ==========================================================
# Check Missing Values
# ==========================================================

print(sales_fe[rolling_columns].isna().sum())

,date,customers,customer_rolling_mean_7,customer_rolling_std_7,customer_rolling_mean_14,customer_rolling_std_14,customer_rolling_mean_21,customer_rolling_std_21,customer_rolling_mean_28,customer_rolling_std_28
0,2013-01-07,785,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2013-01-08,654,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2013-01-09,626,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2013-01-10,615,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2013-01-11,592,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,2013-01-12,646,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,2013-01-13,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,2013-01-14,616,559.714286,254.578550,NaN,NaN,NaN,NaN,NaN,NaN
8,2013-01-15,512,535.571429,237.063885,NaN,NaN,NaN,NaN,NaN,NaN
9,2013-01-16,530,515.285714,231.244974,NaN,NaN,NaN,NaN,NaN,NaN


customer_rolling_mean_7      4732
customer_rolling_std_7       4732
customer_rolling_mean_14     9464
customer_rolling_std_14      9464
customer_rolling_mean_21    14196
customer_rolling_std_21     14196
customer_rolling_mean_28    18928
customer_rolling_std_28     18928
dtype: int64


In [7]:
# ==========================================================
# Generate Historical Features for Future Dataset
# ==========================================================
# Use the latest historical information of each store to
# create lag and rolling customer features for future
# customer prediction.
#
# Future customer values are unknown, therefore only
# historical customer information is used.
# Recursive forecasting is not applied.
# ==========================================================

lag_features = [1, 7, 14, 21, 28]
rolling_windows = [7, 14, 21, 28]

# ==========================================================
# Sort Historical Data
# ==========================================================

sales_fe = (
    sales_fe
    .sort_values(["store_id", "date"])
    .reset_index(drop=True)
)

for store in future_fe["store_id"].unique():

    history = (
        sales_fe.loc[
            sales_fe["store_id"] == store
            ]
        .sort_values("date")
    )

    # ======================================================
    # Customer Lag Features
    # ======================================================

    for lag in lag_features:
        future_fe.loc[
            future_fe["store_id"] == store,
            f"customer_lag_{lag}"
        ] = history["customers"].iloc[-lag]

    # ======================================================
    # Promotion Lag Features
    # ======================================================

    for lag in [1, 7]:
        future_fe.loc[
            future_fe["store_id"] == store,
            f"promo_lag_{lag}"
        ] = history["promo"].iloc[-lag]

    # ======================================================
    # Customer Rolling Features
    # ======================================================

    for window in rolling_windows:
        future_fe.loc[
            future_fe["store_id"] == store,
            f"customer_rolling_mean_{window}"
        ] = (
            history["customers"]
            .tail(window)
            .mean()
        )

        future_fe.loc[
            future_fe["store_id"] == store,
            f"customer_rolling_std_{window}"
        ] = (
            history["customers"]
            .tail(window)
            .std()
        )

    # ======================================================
    # Promotion Rolling Features
    # ======================================================

    for window in [7, 14]:
        future_fe.loc[
            future_fe["store_id"] == store,
            f"promo_mean_{window}"
        ] = (
            history["promo"]
            .tail(window)
            .mean()
        )

In [8]:
print(
    future_fe[
        [
            "customer_lag_1",
            "customer_lag_7",
            "customer_rolling_mean_7",
            "promo_lag_1"
        ]
    ].head()
)

print(
    future_fe[
        [
            "customer_lag_1",
            "customer_lag_7",
            "customer_rolling_mean_7",
            "promo_lag_1",
            "promo_mean_7"
        ]
    ].describe()
)

print(
    future_fe[
        [
            "customer_lag_1",
            "customer_lag_7",
            "customer_rolling_mean_7",
            "promo_lag_1",
            "promo_mean_7"
        ]
    ].isna().sum()
)

   customer_lag_1  customer_lag_7  customer_rolling_mean_7  promo_lag_1
0             0.0           553.0               456.428571          0.0
1             0.0           812.0               621.142857          0.0
2             0.0          1252.0               909.428571          0.0
3             0.0           935.0               670.142857          0.0
4             0.0           759.0               550.285714          0.0
       customer_lag_1  customer_lag_7  customer_rolling_mean_7  promo_lag_1  \
count    40560.000000    40560.000000             40560.000000      40560.0   
mean        56.958580      897.511834               683.109679          0.0   
std        372.477264      396.966186               369.015133          0.0   
min          0.000000      309.000000               221.857143          0.0   
25%          0.000000      671.500000               491.392857          0.0   
50%          0.000000      813.000000               601.714286          0.0   
75%          0.

In [9]:
#interaction feature
sales_fe["promo_weekend"] = (
    sales_fe["promo"] *
    sales_fe["is_weekend"]
)

future_fe["promo_weekend"] = (
    future_fe["promo"] *
    future_fe["is_weekend"]
)

sales_fe["promo_schoolholiday"] = (
    sales_fe["promo"] *
    sales_fe["school_holiday"]
)

future_fe["promo_schoolholiday"] = (
    future_fe["promo"] *
    future_fe["school_holiday"]
)

sales_fe["promo_stateholiday"] = (
    sales_fe["promo"] *
    (sales_fe["state_holiday"] != "0").astype(int)
)

future_fe["promo_stateholiday"] = (
    future_fe["promo"] *
    (future_fe["state_holiday"] != "0").astype(int)
)

In [10]:
# ==========================================================
# One-Hot Encoding
# ==========================================================
# Tree-based machine learning models require numerical inputs.
# Convert categorical variables into binary indicator variables
# using one-hot encoding.
#
# Features:
# store_type
# assortment
# state_holiday
# ==========================================================

categorical_features = [
    "store_type",
    "assortment",
    "state_holiday"
]

sales_fe = pd.get_dummies(
    sales_fe,
    columns=categorical_features,
    dtype=int
)

future_fe = pd.get_dummies(
    future_fe,
    columns=categorical_features,
    dtype=int
)

# ==========================================================
# Align Training and Future Features
# ==========================================================
# Ensure both datasets contain exactly the same feature columns.
# Missing columns in the future dataset are filled with zeros.

sales_fe, future_fe = sales_fe.align(
    future_fe,
    join="left",
    axis=1,
    fill_value=0
)

# Restore the target variable after alignment.

sales_fe["customers"] = sales_clean["customers"]
# ==========================================================
# Validate One-Hot Encoding
# ==========================================================

encoded_columns = [
    col for col in sales_fe.columns
    if col.startswith("store_type_")
       or col.startswith("assortment_")
       or col.startswith("state_holiday_")
]

print("=" * 60)
print("ONE-HOT ENCODED FEATURES")
print("=" * 60)

print(encoded_columns)

print("\nTraining sample")
display(sales_fe[encoded_columns].head())

print("\nFuture sample")
display(future_fe[encoded_columns].head())

print("\nMissing values (Training)")
print(sales_fe[encoded_columns].isna().sum())

print("\nMissing values (Future)")
print(future_fe[encoded_columns].isna().sum())

ONE-HOT ENCODED FEATURES
['store_type_a', 'store_type_b', 'store_type_c', 'store_type_d', 'assortment_a', 'assortment_b', 'assortment_c', 'state_holiday_0', 'state_holiday_a', 'state_holiday_b', 'state_holiday_c']

Training sample


,store_type_a,store_type_b,store_type_c,store_type_d,assortment_a,assortment_b,assortment_c,state_holiday_0,state_holiday_a,state_holiday_b,state_holiday_c
0,0,0,1,0,1,0,0,1,0,0,0
1,0,0,1,0,1,0,0,1,0,0,0
2,0,0,1,0,1,0,0,1,0,0,0
3,0,0,1,0,1,0,0,1,0,0,0
4,0,0,1,0,1,0,0,1,0,0,0



Future sample


,store_type_a,store_type_b,store_type_c,store_type_d,assortment_a,assortment_b,assortment_c,state_holiday_0,state_holiday_a,state_holiday_b,state_holiday_c
0,0,0,1,0,1,0,0,1,0,0,0
1,1,0,0,0,1,0,0,1,0,0,0
2,1,0,0,0,0,0,1,1,0,0,0
3,1,0,0,0,1,0,0,1,0,0,0
4,1,0,0,0,0,0,1,1,0,0,0



Missing values (Training)
store_type_a       0
store_type_b       0
store_type_c       0
store_type_d       0
assortment_a       0
assortment_b       0
assortment_c       0
state_holiday_0    0
state_holiday_a    0
state_holiday_b    0
state_holiday_c    0
dtype: int64

Missing values (Future)
store_type_a       0
store_type_b       0
store_type_c       0
store_type_d       0
assortment_a       0
assortment_b       0
assortment_c       0
state_holiday_0    0
state_holiday_a    0
state_holiday_b    0
state_holiday_c    0
dtype: int64


In [11]:
# ==========================================================
# Final Feature Validation
# ==========================================================
# Perform a final check before modelling.

print("=" * 60)
print("FINAL FEATURE VALIDATION")
print("=" * 60)

print("\nTraining shape:")
print(sales_fe.shape)

print("\nFuture shape:")
print(future_fe.shape)

print("\nTraining missing values:")
print(sales_fe.isna().sum().sort_values(ascending=False).head(20))

print("\nFuture missing values:")
print(future_fe.isna().sum().sort_values(ascending=False).head(20))

print("\nTraining columns:")
print(sales_fe.columns.tolist())

print("\nFuture columns:")
print(future_fe.columns.tolist())

FINAL FEATURE VALIDATION

Training shape:
(624624, 49)

Future shape:
(40560, 49)

Training missing values:
customer_rolling_mean_28    18928
customer_lag_28             18928
customer_rolling_std_28     18928
customer_rolling_mean_21    14196
customer_rolling_std_21     14196
customer_lag_21             14196
customer_rolling_std_14      9464
customer_rolling_mean_14     9464
customer_lag_14              9464
promo_mean_14                9464
promo_lag_7                  4732
customer_rolling_std_7       4732
customer_rolling_mean_7      4732
promo_mean_7                 4732
customer_lag_7               4732
customer_lag_1                676
promo_lag_1                   676
assortment_c                    0
state_holiday_b                 0
state_holiday_a                 0
dtype: int64

Future missing values:
customers                   40560
store_id                        0
promo_stateholiday              0
customer_rolling_mean_14        0
customer_rolling_std_14         0
custo

In [12]:
# ==========================================================
# Convert Store ID to Integer
# ==========================================================

sales_fe["store_id"] = (
    sales_fe["store_id"]
    .str.replace("store_", "")
    .astype(int)
)

future_fe["store_id"] = (
    future_fe["store_id"]
    .str.replace("store_", "")
    .astype(int)
)

# ==========================================================
# Save Feature Engineered Datasets
# ==========================================================

sales_fe.to_csv(
    processed_dir / "customer_fe.csv",
    index=False
)

future_fe.to_csv(
    processed_dir / "future_customer_fe.csv",
    index=False
)

# ==========================================================
# Validate Future Features
# ==========================================================

print(
    future_fe[
        [
            "customer_lag_1",
            "customer_lag_7",
            "customer_rolling_mean_7",
            "promo_lag_1",
            "promo_mean_7"
        ]
    ].describe()
)

# ==========================================================
# Validate Historical Customer Information
# ==========================================================

print(
    sales_fe
    .groupby("store_id")["customers"]
    .last()
    .describe()
)

print(
    sales_fe
    .groupby("store_id")["promo"]
    .last()
    .value_counts()
)

print(
    sales_fe["store_id"].nunique()
)

print(
    sales_fe
    .groupby("store_id")[["customers", "open"]]
    .last()
    .head(20)
)

       customer_lag_1  customer_lag_7  customer_rolling_mean_7  promo_lag_1  \
count    40560.000000    40560.000000             40560.000000      40560.0   
mean        56.958580      897.511834               683.109679          0.0   
std        372.477264      396.966186               369.015133          0.0   
min          0.000000      309.000000               221.857143          0.0   
25%          0.000000      671.500000               491.392857          0.0   
50%          0.000000      813.000000               601.714286          0.0   
75%          0.000000     1021.750000               755.214286          0.0   
max       4691.000000     3592.000000              3514.285714          0.0   

       promo_mean_7  
count  4.056000e+04  
mean   7.142857e-01  
std    2.220473e-16  
min    7.142857e-01  
25%    7.142857e-01  
50%    7.142857e-01  
75%    7.142857e-01  
max    7.142857e-01  
count     676.000000
mean      661.109467
std       520.593840
min         0.000000
25%   

In [13]:
print(sales_fe.columns.tolist())

['store_id', 'date', 'sales', 'customers', 'open', 'promo', 'school_holiday', 'competition_distance', 'year', 'month', 'quarter', 'week_of_year', 'day_of_week', 'day_of_month', 'is_weekend', 'is_month_start', 'is_month_end', 'is_quarter_end', 'customer_lag_1', 'customer_lag_7', 'customer_lag_14', 'customer_lag_21', 'customer_lag_28', 'promo_lag_1', 'promo_lag_7', 'customer_rolling_mean_7', 'customer_rolling_std_7', 'customer_rolling_mean_14', 'customer_rolling_std_14', 'customer_rolling_mean_21', 'customer_rolling_std_21', 'customer_rolling_mean_28', 'customer_rolling_std_28', 'promo_mean_7', 'promo_mean_14', 'promo_weekend', 'promo_schoolholiday', 'promo_stateholiday', 'store_type_a', 'store_type_b', 'store_type_c', 'store_type_d', 'assortment_a', 'assortment_b', 'assortment_c', 'state_holiday_0', 'state_holiday_a', 'state_holiday_b', 'state_holiday_c']
